# PhoBERT -- Single-augmentation experiments (EDA / BT / LLM)

## Dependencies

In [ ]:
!pip install -q transformers datasets huggingface_hub scikit-learn accelerate sentencepiece py_vncorenlp
!pip install -q -U datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.4 MB/s eta 0:00:00


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU in Runtime > Change runtime type.")

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
user_info = whoami()
print(f"Authenticated as: {user_info['name']}")

Authenticated as: AnoraLee


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUGMENTED_DIR = DATA_DIR / "augmented"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [ ]:
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 128
SEED = 42
LABELS = ["CLEAN", "OFFENSIVE", "HATE"]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

## Chuẩn hoá & Phân từ dùng chung

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import py_vncorenlp

def text_key(text):
    """Normalize a text for exact-duplicate / leakage matching."""
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

def add_key(df, source_col="text_raw"):
    df = df.copy()
    df["_key"] = df[source_col].map(text_key)
    return df

_vncorenlp_dir = "/content/vncorenlp"
if not os.path.exists(_vncorenlp_dir):
    os.makedirs(_vncorenlp_dir, exist_ok=True)
    py_vncorenlp.download_model(save_dir=_vncorenlp_dir)
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=_vncorenlp_dir)

def segment_text(text):
    try:
        sentences = rdrsegmenter.word_segment(str(text))
        return " ".join(sentences)
    except Exception:
        return str(text)

def apply_segmentation(df, source_col="text_raw", target_col="text"):
    df = df.copy()
    df[target_col] = df[source_col].map(segment_text)
    return df

In [ ]:
TOXIC_TEENCODE_MAP = {
    r"\bko\b": "không", r"\bhok\b": "không", r"\bdc\b": "được", r"\bđc\b": "được",
    r"\bj\b": "gì", r"\bbt\b": "bình thường", r"\btrc\b": "trước", r"\bnhg\b": "nhưng",
    r"\bthg\b": "thằng",
    r"\bdm\b": "địt mẹ", r"\bđm\b": "địt mẹ", r"\bdkm\b": "địt con mẹ", r"\bđkm\b": "địt con mẹ",
    r"\bvkl\b": "vãi lồn", r"\bvcl\b": "vãi lồn", r"\bvl\b": "vãi lồn", r"\bkl\b": "cái lồn",
    r"\bcc\b": "cục cứt", r"\bcđm\b": "cộng đồng mạng", r"\bml\b": "mặt lồn",
    r"\bđjt\b": "địt", r"\bdjt\b": "địt", r"\bdit\b": "địt",
    r"\bloz\b": "lồn", r"\blon\b": "lồn",
    r"\bcac\b": "cặc", r"\bcặk\b": "cặc", r"\bđb\b": "đầu buồi",
    r"\bcút\b": "cút", r"\bđĩ\b": "đĩ", r"\bphò\b": "phò"
}

def normalize_teencode(text):
    for pattern, replacement in TOXIC_TEENCODE_MAP.items():
        text = re.sub(pattern, replacement, str(text), flags=re.IGNORECASE)
    return text

def apply_teencode_normalization(df, col="text_raw"):
    df = df.copy()
    df[col] = df[col].apply(normalize_teencode)
    return df

## Load dev/test đã đóng băng

In [ ]:
dev_raw = pd.read_csv(PROCESSED_DIR / "dev.csv")
test_raw = pd.read_csv(PROCESSED_DIR / "test.csv")

dev_raw = dev_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in dev_raw.columns else dev_raw
test_raw = test_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in test_raw.columns else test_raw

dev_df = apply_teencode_normalization(dev_raw.dropna(subset=["text_raw"]).reset_index(drop=True))
test_df = apply_teencode_normalization(test_raw.dropna(subset=["text_raw"]).reset_index(drop=True))

dev_keys = set(dev_df["text_raw"].map(text_key))
test_keys = set(test_df["text_raw"].map(text_key))

dev_df = apply_segmentation(dev_df)
test_df = apply_segmentation(test_df)

print(f"Validation: {dev_df.shape}")
print(f"Test: {test_df.shape}")

Validation: (2650, 5)
Test: (6576, 5)


## Tokenizer + hàm dùng chung

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_split(df):
    output = df[["text", "label"]].dropna().copy()
    assert output["label"].isin(LABELS).all(), "Unexpected label found."
    output["label"] = output["label"].map(label2id)
    return Dataset.from_pandas(output, preserve_index=False)

def build_tokenized_dataset(train_df, dev_df, test_df):
    dataset = DatasetDict({
        "train": prepare_split(train_df),
        "validation": prepare_split(dev_df),
        "test": prepare_split(test_df),
    })
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    model_columns = [
        c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"]
        if c in tokenized["train"].column_names
    ]
    tokenized.set_format("torch", columns=model_columns)
    print(tokenized)
    return tokenized

## Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Unified metric schema -- MUST stay identical across all 3 notebooks
    so results in the report are directly comparable."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "hate_f1": f1_score(labels, predictions, labels=[LABELS.index("HATE")], average="macro"),
    }

## Train (hàm dùng chung)

In [ ]:
import json
from transformers import TrainingArguments, AutoModelForSequenceClassification, Trainer, set_seed

set_seed(SEED)

def train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=None):
    """Load a FRESH PhoBERT model and train one experiment end to end.
    Always instantiates a new model -- never reuses a model/trainer object
    from a previous cell, to avoid accidental weight leakage between runs."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, label2id=label2id, id2label=id2label,
    )
    model_dir = MODELS_DIR / model_dir_name

    training_args = TrainingArguments(
        output_dir=str(model_dir / "_checkpoints"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{experiment_name}] test metrics: {test_metrics}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    experiment_results_dir = RESULTS_DIR / experiment_name
    experiment_results_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = experiment_results_dir / f"metrics_{experiment_name}.json"
    with open(metrics_path, "w") as f:
        json.dump(test_metrics, f, indent=2)

    print(f"Model saved to: {model_dir}")
    print(f"Metrics saved to: {metrics_path}")

    if push_to_hub_repo:
        api = HfApi()
        api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=True, exist_ok=True)
        trainer.model.push_to_hub(push_to_hub_repo, private=True,
                                  commit_message=f"Upload PhoBERT {experiment_name} model")
        tokenizer.push_to_hub(push_to_hub_repo, private=True,
                              commit_message="Upload PhoBERT tokenizer")
        api.upload_file(
            path_or_fileobj=str(metrics_path),
            path_in_repo="test_metrics.json",
            repo_id=push_to_hub_repo, repo_type="model",
        )
        print(f"Uploaded: https://huggingface.co/{push_to_hub_repo}")

    return trainer, test_metrics

## Hàm load + làm sạch + chạy 1 thí nghiệm

In [ ]:
def load_and_clean_source(csv_path, dev_keys, test_keys, combine_with_baseline=False):
    raw = pd.read_csv(csv_path)
    if combine_with_baseline:
        baseline = pd.read_csv(AUGMENTED_DIR / "aug_baseline.csv")
        raw = pd.concat([baseline, raw], ignore_index=True)

    raw = apply_teencode_normalization(raw)
    raw = add_key(raw)

    conflict_keys = set(
        raw.groupby("_key")["label"].nunique().loc[lambda c: c > 1].index
    )
    blocked_keys = conflict_keys | dev_keys | test_keys
    clean = (
        raw[~raw["_key"].isin(blocked_keys)]
        .drop_duplicates("_key")
        .drop(columns="_key")
        .reset_index(drop=True)
    )
    print(f"  {csv_path.name}: {len(raw):,} -> {len(clean):,} dòng "
          f"(loại {len(conflict_keys)} nhãn xung đột + leakage)")
    return clean

In [ ]:
def run_single_augmentation_experiment(experiment_name, model_dir_name, source_specs, push=True):
    parts = [load_and_clean_source(p, dev_keys, test_keys, combine_with_baseline=c)
             for p, c in source_specs]
    train_df = pd.concat(parts, ignore_index=True).drop_duplicates("text_raw").reset_index(drop=True)
    print(f"  Tổng train: {len(train_df):,}")
    print(f"  {train_df['label'].value_counts().to_dict()}")

    train_df = apply_segmentation(train_df)
    tokenized = build_tokenized_dataset(train_df, dev_df, test_df)

    repo = f"{whoami()['name']}/vietnamese-hsd-phobert-{experiment_name}" if push else None
    return train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=repo)

## Experiment 1 -- EDA

In [ ]:
eda_trainer, eda_metrics = run_single_augmentation_experiment(
    experiment_name="eda",
    model_dir_name="eda_phobert",
    source_specs=[(AUGMENTED_DIR / "aug_eda.csv", False)],
)

  aug_eda.csv: 134,412 -> 133,439 dòng (loại 44 nhãn xung đột + leakage)
  Tổng train: 133,439
  {'CLEAN': 59266, 'OFFENSIVE': 57439, 'HATE': 16734}


Map:   0%|          | 0/133439 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 133439
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Hate F1
0.268908,0.566266,3,0.784367,0.582761,0.813251,0.560000


[eda] test metrics: {'test_loss': 0.5662659406661987, 'test_accuracy': 0.7843673965936739, 'test_macro_f1': 0.5827606500877253, 'test_weighted_f1': 0.8132514296998178, 'test_hate_f1': 0.56}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/Hate_Speech_Detection/models/eda_phobert
Metrics saved to: /content/drive/MyDrive/Hate_Speech_Detection/results/eda/metrics_eda.json


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6aa54a79-73e3d6611ec381055c6a22a5;8933aa5b-17f2-4424-8aa5-75c1ee022f3a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

## Experiment 2 -- Back-translation

In [ ]:
bt_trainer, bt_metrics = run_single_augmentation_experiment(
    experiment_name="bt",
    model_dir_name="bt_phobert",
    source_specs=[(AUGMENTED_DIR / "aug_bt.csv", True)],
)

  aug_bt.csv: 89,361 -> 88,509 dòng (loại 12 nhãn xung đột + leakage)
  Tổng train: 88,509
  {'CLEAN': 59276, 'OFFENSIVE': 22828, 'HATE': 6405}


Map:   0%|          | 0/88509 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 88509
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: https://huggingface.co/AnoraLee/vietnamese-hsd-phobert-bt


## Experiment 3 -- LLM-generated

In [ ]:
llm_trainer, llm_metrics = run_single_augmentation_experiment(
    experiment_name="llm",
    model_dir_name="llm_phobert",
    source_specs=[
        (AUGMENTED_DIR / "aug_llm.csv", False),
    ],
)

  aug_llm.csv: 88,191 -> 87,339 dòng (loại 12 nhãn xung đột + leakage)
  Tổng train: 87,339
  {'CLEAN': 59276, 'OFFENSIVE': 20455, 'HATE': 7608}


Map:   0%|          | 0/87339 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 87339
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded: https://huggingface.co/AnoraLee/vietnamese-hsd-phobert-llm
